# 21 规则置入置出分析：真实放款数据进阶案例

使用 `hscredit_yyp.xlsx`，以“衡枢鉴真分老客版”对应字段为主评分、`FPD` 为实际标签，
演示真实数据上的完整漏斗、金额、有偏通过率、多逾期标签和多评分。

> 原始 Excel 的部分字段名在历史样例文件中存在编码问题，本示例按稳定列位置映射为可读别名。

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from IPython.display import display

from hscredit.core.rules import Rule
from hscredit.report import feature_bin_stats, rule_swap_analysis

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

## 1. 读取并建立清晰字段别名

In [2]:
raw = pd.read_excel('hscredit_yyp.xlsx')

data = pd.DataFrame({
    'apply_date': pd.to_datetime(raw.iloc[:, 1]),
    'amount': raw.iloc[:, 2],
    'MOB1': raw['MOB1'],
    'score_alt': raw.iloc[:, 6],                 # 中智小牛分 C3
    'fraud_score': raw.iloc[:, 8].fillna(-1),    # 反欺诈分
    'qingyun24': raw.iloc[:, 9],
    'multihead6m': raw.iloc[:, 13],
    'score_main': raw.iloc[:, 16],               # 衡枢鉴真分老客版
    'target': raw['FPD'].astype(int),
})

print(f"样本数: {len(data):,}")
print(f"FPD 坏样本率: {data['target'].mean():.2%}")
display(data[['score_main', 'score_alt', 'fraud_score', 'qingyun24',
              'multihead6m', 'amount', 'target']].describe().T)

样本数: 970
FPD 坏样本率: 14.02%


,count,mean,std,min,25%,50%,75%,max
score_main,970.0000,0.0946,0.0524,0.0095,0.0541,0.0838,0.1239,0.3076
score_alt,307.0000,635.9316,91.2117,464.0000,564.5000,629.0000,704.0000,850.0000
fraud_score,970.0000,-0.5847,0.6243,-1.0000,-1.0000,-1.0000,0.0884,0.8808
qingyun24,970.0000,604.3258,64.9345,372.0000,561.2500,603.0000,647.0000,850.0000
multihead6m,970.0000,60.8526,12.1311,19.0000,52.0000,61.0000,69.0000,94.0000
amount,970.0000,4213.6113,2026.4627,244.0000,2942.7500,4295.0000,5750.0000,8930.0000
target,970.0000,0.1402,0.3474,0.0000,0.0000,0.0000,0.0000,1.0000


## 2. 规则集与主评分风险分箱

In [3]:
rules_base = [
    Rule('fraud_score >= 0.70', name='生产反欺诈拒绝'),
    Rule('score_main >= 0.23', name='生产高风险拒绝'),
]
rules_out = [
    Rule('multihead6m >= 75', name='本次置出高多头'),
    Rule('qingyun24 < 520', name='本次置出低青云分'),
]
rules_in = [
    Rule('(qingyun24 >= 680) & (score_main < 0.10)', name='本次置入高青云低风险'),
    Rule('(multihead6m <= 50) & (score_main < 0.06)', name='本次置入低多头低风险'),
]

main_bin_table = feature_bin_stats(
    data,
    feature='score_main',
    target='target',
    method='quantile',
    max_n_bins=8,
    margins=True,
    n_jobs=1,
)
display(main_bin_table)

,指标名称,指标含义,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,风险拒绝比,累积LIFT值,累积坏账改善,累计风险拒绝比,累积好样本数,累积坏样本数,分档KS值
0,score_main,score_main,"[-inf, 0.0400)",121,110,11,0.1247,0.1319,0.0809,0.0909,-0.4890,0.0249,0.2804,0.6484,-0.0501,-0.4017,1.0000,1.0000,1.0000,834,136,0.0000
1,score_main,score_main,"[0.0400, 0.0541)",122,113,9,0.1258,0.1355,0.0662,0.0738,-0.7166,0.0497,0.2804,0.5262,-0.0682,-0.5420,1.0501,0.3516,0.4017,724,125,0.0510
2,score_main,score_main,"[0.0541, 0.0697)",121,109,12,0.1247,0.1307,0.0882,0.0992,-0.3929,0.0167,0.2804,0.7073,-0.0417,-0.3344,1.1380,0.4130,0.5510,611,116,0.1203
3,score_main,score_main,"[0.0697, 0.0838)",121,99,22,0.1247,0.1187,0.1618,0.1818,0.3095,0.0133,0.2804,1.2968,0.0423,0.3391,1.2240,0.3730,0.5970,502,104,0.1628
4,score_main,score_main,"[0.0838, 0.1023)",122,108,14,0.1258,0.1295,0.1029,0.1148,-0.2295,0.0061,0.2804,0.8185,-0.0261,-0.2077,1.2059,0.2059,0.4118,403,82,0.1197
5,score_main,score_main,"[0.1023, 0.1239)",120,105,15,0.1237,0.1259,0.1103,0.1250,-0.1323,0.0021,0.2804,0.8915,-0.0153,-0.1238,1.3361,0.2010,0.5371,295,68,0.1463
6,score_main,score_main,"[0.1239, 0.1581)",121,105,16,0.1247,0.1259,0.1176,0.1322,-0.0678,0.0006,0.2804,0.9431,-0.0081,-0.0650,1.5556,0.1857,0.7413,190,53,0.1619
7,score_main,score_main,"[0.1581, +inf)",122,85,37,0.1258,0.1019,0.2721,0.3033,0.9818,0.1671,0.2804,2.1631,0.1673,1.3304,2.1631,0.1673,1.3304,85,37,0.1701
8,score_main,score_main,合计,970,834,136,1.0000,1.0000,1.0000,0.1402,0.0000,0.0000,0.2804,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,834,136,0.1701


## 3. 完整置换：独立规则、68% 样本幸存率、订单与金额双口径

In [4]:
result = rule_swap_analysis(
    data=data,
    score='score_main',
    rules_base=rules_base,
    rules_out=rules_out,
    rules_in=rules_in,
    bin_table=main_bin_table,
    target='target',
    amount='amount',
    sample_survival_rate=0.68,
    out_in_uplift=2.0,
    rule_analysis_mode='independent',
    n_jobs=1,
)
pipeline = result['swap_pipeline']

view_columns = [
    '规则分类', '指标名称', '风险来源', '样本总数', '样本占比',
    '原始坏样本率', '调整后坏样本率', '生产通过率',
    '阶段前样本数', '阶段后样本数', '样本总额', '金额坏样本率',
]
display(pipeline[[column for column in view_columns if column in pipeline.columns]])

,规则分类,指标名称,风险来源,样本总数,样本占比,原始坏样本率,调整后坏样本率,生产通过率,阶段前样本数,阶段后样本数,样本总额
0,全量样本,,实际表现+评分预测,970,1.0000,0.1446,0.1593,68.0000,970,970,4087203.0000
1,OUT-OUT拒绝,生产反欺诈拒绝,实际表现,27,0.0278,0.2593,0.2593,66.1072,970,943,96603.0000
2,OUT-OUT拒绝,生产高风险拒绝,实际表现,20,0.0206,0.4500,0.4500,66.5979,970,950,91192.0000
3,OUT-OUT拒绝,合计,实际表现,46,0.0474,0.3261,0.3261,64.7753,970,924,181038.0000
4,剩余样本,,实际表现+评分预测,924,0.9526,0.1355,0.1509,64.7753,924,924,3906165.0000
5,IN-OUT置出,本次置出高多头,实际表现,124,0.1278,0.2177,0.2177,56.0825,924,800,517349.0000
6,IN-OUT置出,本次置出低青云分,实际表现,85,0.0876,0.1529,0.1529,58.8165,924,839,307927.0000
7,IN-OUT置出,合计,实际表现,194,0.2000,0.1959,0.1959,51.1753,924,730,769326.0000
8,total通过样本,,实际表现+评分预测,730,0.7526,0.1195,0.1390,51.1753,730,730,3136839.0000
9,IN-IN通过,,实际表现,585,0.6031,0.1248,0.1248,41.0103,730,585,2459135.0000


In [5]:
def stage_row(frame, category):
    rows = frame[frame['规则分类'] == category]
    totals = rows[rows['指标名称'].fillna('') == '合计']
    return (totals if not totals.empty else rows).iloc[-1]


base_union = np.logical_or.reduce([rule.predict(data).to_numpy() for rule in rules_base])
after_base = ~base_union
out_union = after_base & np.logical_or.reduce([rule.predict(data).to_numpy() for rule in rules_out])
total_pass = after_base & ~out_union
in_union = total_pass & np.logical_or.reduce([rule.predict(data).to_numpy() for rule in rules_in])
in_in = total_pass & ~in_union

assert stage_row(pipeline, 'OUT-OUT拒绝')['样本总数'] == base_union.sum()
assert stage_row(pipeline, 'IN-OUT置出')['样本总数'] == out_union.sum()
assert stage_row(pipeline, 'OUT-IN置入')['样本总数'] == in_union.sum()
assert stage_row(pipeline, 'IN-IN通过')['样本总数'] == in_in.sum()
assert stage_row(pipeline, 'ALL-IN置换')['样本总数'] == total_pass.sum()
assert base_union.sum() + out_union.sum() + in_in.sum() + in_union.sum() == len(data)
assert pipeline.iloc[0]['生产通过率'] == 68.0

print('真实样本的四个互斥客群守恒，生产漏斗起点已校准为 68%。')

真实样本的四个互斥客群守恒，生产漏斗起点已校准为 68%。


## 4. 置换前后结果与 OUT-IN 预测风险

In [6]:
display(result['swap_result'])

risk_rows = pipeline.loc[
    pipeline['规则分类'].isin(['IN-IN通过', 'OUT-IN置入']),
    ['规则分类', '指标名称', '风险来源', '原始坏样本数', '调整后坏样本数',
     '原始坏样本率', '调整后坏样本率'],
]
display(risk_rows)

assert '实际表现' in stage_row(pipeline, 'IN-IN通过')['风险来源']
assert '评分预测' in stage_row(pipeline, 'OUT-IN置入')['风险来源']

,指标,变化前,变化后,绝对变化,相对变化
0,通过率,41.0103,51.1753,10.1649,0.2479
1,逾期率,0.1248,0.1390,0.0142,0.1139
2,风险上浮系数,1.0000,1.1632,0.1632,0.1632
3,样本集幸存比例,0.6800,0.6800,0.0000,0.0000


,规则分类,指标名称,风险来源,原始坏样本数,调整后坏样本数,原始坏样本率,调整后坏样本率
9,IN-IN通过,,实际表现,73.0000,73.0000,0.1248,0.1248
10,OUT-IN置入,本次置入高青云低风险,评分预测,7.2703,14.5406,0.1136,0.2272
11,OUT-IN置入,本次置入低多头低风险,评分预测,7.5342,15.0684,0.0856,0.1712
12,OUT-IN置入,合计,评分预测,14.2367,28.4733,0.0982,0.1964


## 5. 可选：让 IN-IN 也应用情景上浮

In [7]:
stressed = rule_swap_analysis(
    data=data,
    score='score_main',
    rules_base=rules_base,
    rules_out=rules_out,
    rules_in=rules_in,
    bin_table=main_bin_table,
    target='target',
    risk_uplifts={'in_in': 1.10},
    n_jobs=1,
)['swap_pipeline']

stressed_in_in = stage_row(stressed, 'IN-IN通过')
display(stressed.loc[stressed['规则分类'] == 'IN-IN通过',
                     ['规则分类', '风险来源', '原始坏样本数', '调整后坏样本数']])
assert np.isclose(
    stressed_in_in['调整后坏样本数'],
    stressed_in_in['原始坏样本数'] * 1.10,
)

,规则分类,风险来源,原始坏样本数,调整后坏样本数
9,IN-IN通过,实际表现,73.0000,80.3000


## 6. 多逾期标签：MOB1 DPD7 / DPD3 / DPD0

In [8]:
mob1_bin_table = feature_bin_stats(
    data,
    feature='score_main',
    overdue='MOB1',
    dpds=[7, 3, 0],
    method='quantile',
    max_n_bins=8,
    margins=True,
    n_jobs=1,
)
mob1_result = rule_swap_analysis(
    data=data,
    score='score_main',
    rules_base=rules_base,
    rules_out=rules_out,
    rules_in=rules_in,
    bin_table=mob1_bin_table,
    overdue='MOB1',
    dpds=[7, 3, 0],
    amount='amount',
    n_jobs=1,
)

mob1_pipeline = mob1_result['swap_pipeline']
display(mob1_pipeline)
assert isinstance(mob1_pipeline.columns, pd.MultiIndex)
assert {'MOB1_7+', 'MOB1_3+', 'MOB1_0+'}.issubset(set(mob1_pipeline.columns.get_level_values(0)))
assert mob1_pipeline[('MOB1_7+', '调整后坏样本率')].max() > 0

分箱详情                                                                                                                                                            ... MOB1_0+                                                                                                                               
         规则分类        指标名称                                       规则详情   行类型 样本总数   样本占比 阶段前样本数 阶段后样本数 阶段前生产通过率    生产通过率      通过率 通过率(绝对值) 通过率(相对值)    通过率变化         样本总额  ...   好样本占比     坏样本数  坏样本占比   坏样本率   原始坏样本数 原始坏样本率  调整后坏样本数 调整后坏样本率  LIFT值    坏账改善    风险拒绝比     原始坏样本总额    调整后坏样本总额 原始坏样本率(金额) 调整后坏样本率(金额)
0        全量样本                                                           状态  970 1.0000    970    970 100.0000 100.0000 100.0000 100.0000   0.0000   0.0000 4087203.0000  ...  0.7674 225.6650 0.2326 0.2326 200.8325 0.2070 225.6650  0.2326 1.0000  0.0000   0.0000 846218.4185 963597.8370     0.2070      0.2358
1   OUT-OUT拒绝     生产反欺诈拒绝                        fraud_score >= 0.70  规则明细   27 0.0278    970    943 100.0000  97.2165  97.2165  97.2165  -0.0278  -2.7835   96603.0000  ...  0.7037   8.0000 0.2963 0.2963   8.0000 0.2963   8.0000  0.2963 1.2736 -0.2736  -9.8294  30905.0000  30905.0000     0.3199      0.3199
2   OUT-OUT拒绝     生产高风险拒绝                         score_main >= 0.23  规则明细   20 0.0206    970    950 100.0000  97.9381  97.9381  97.9381  -0.0206  -2.0619   91192.0000  ...  0.4500  11.0000 0.5500 0.5500  11.0000 0.5500  11.0000  0.5500 2.3641 -1.3641 -66.1600  50498.0000  50498.0000     0.5538      0.5538
3   OUT-OUT拒绝          合计                                             阶段合计   46 0.0474    970    924 100.0000  95.2577  95.2577  95.2577  -0.0474  -4.7423  181038.0000  ...  0.6087  18.0000 0.3913 0.3913  18.0000 0.3913  18.0000  0.3913 1.6820 -0.6820 -14.3810  74646.0000  74646.0000     0.4123      0.4123
4        剩余样本                                                           状态  924 0.9526    924    924  95.2577  95.2577  95.2577  95.2577   0.0000   0.0000 3906165.0000  ...  0.7753 207.6650 0.2247 0.2247 182.8325 0.1979 207.6650  0.2247 0.9660  0.0340   0.0356 771572.4185 888951.8370     0.1975      0.2276
5    IN-OUT置出     本次置出高多头                          multihead6m >= 75  规则明细  124 0.1278    924    800  95.2577  82.4742  82.4742  82.4742  -0.1342 -12.7835  517349.0000  ...  0.7258  34.0000 0.2742 0.2742  34.0000 0.2742  34.0000  0.2742 1.1786 -0.1786  -1.3971 147217.0000 147217.0000     0.2846      0.2846
6    IN-OUT置出    本次置出低青云分                            qingyun24 < 520  规则明细   85 0.0876    924    839  95.2577  86.4948  86.4948  86.4948  -0.0920  -8.7629  307927.0000  ...  0.7294  23.0000 0.2706 0.2706  23.0000 0.2706  23.0000  0.2706 1.1631 -0.1631  -1.8612  79589.0000  79589.0000     0.2585      0.2585
7    IN-OUT置出          合计                                             阶段合计  194 0.2000    924    730  95.2577  75.2577  75.2577  75.2577  -0.2100 -20.0000  769326.0000  ...  0.7268  53.0000 0.2732 0.2732  53.0000 0.2732  53.0000  0.2732 1.1743 -0.1743  -0.8715 212490.0000 212490.0000     0.2762      0.2762
8   total通过样本                                                           状态  730 0.7526    730    730  75.2577  75.2577  75.2577  75.2577   0.0000   0.0000 3136839.0000  ...  0.7881 154.6650 0.2119 0.2119 129.8325 0.1779 154.6650  0.2119 0.9107  0.0893   0.1187 559082.4185 676461.8370     0.1782      0.2157
9     IN-IN通过                                                           状态  585 0.6031    730    585  75.2577  60.3093  60.3093  60.3093  -0.1986 -14.9485 2459135.0000  ...  0.8205 105.0000 0.1795 0.1795 105.0000 0.1795 105.0000  0.1795 0.7715  0.2285   0.3789 441703.0000 441703.0000     0.1796      0.1796
10   OUT-IN置入  本次置入高青云低风险   (qingyun24 >= 680) & (score_main < 0.10)  规则明细   64 0.0660    585    649  60.3093  66.9072  66.9072  66.9072   0.1094   6.5979  349494.0000  ...  0.6379  23.1713 0.3621 0.3621  11.5856 0.1810  23.1713  0.3621 1.5562 -0.5562  -8.4305  63513.6840 127027.3681     0.1817      0.3635
11   

## 7. 多评分：为主评分和备评分分别提供分箱表

In [9]:
score_map = {'衡枢鉴真': 'score_main', '中智小牛C3': 'score_alt'}
multi_reference = data.dropna(subset=['score_main', 'score_alt']).copy()
score_bin_tables = {
    name: feature_bin_stats(
        multi_reference,
        feature=column,
        target='target',
        method='quantile',
        max_n_bins=8,
        margins=True,
        n_jobs=1,
    )
    for name, column in score_map.items()
}

multi_score_result = rule_swap_analysis(
    data=data,
    score=score_map,
    score_weights={'衡枢鉴真': 0.75, '中智小牛C3': 0.25},
    rules_base=rules_base,
    rules_out=rules_out,
    rules_in=rules_in,
    bin_table=score_bin_tables,
    target='target',
    n_jobs=1,
)

display(multi_score_result['swap_result'])
assert set(multi_score_result) == {'swap_pipeline', 'swap_result'}
assert stage_row(multi_score_result['swap_pipeline'], 'ALL-IN置换')['样本总数'] == total_pass.sum()

,指标,变化前,变化后,绝对变化,相对变化
0,通过率,60.3093,75.2577,14.9485,0.2479
1,逾期率,0.1248,0.1394,0.0146,0.1174
2,风险上浮系数,1.0000,1.1647,0.1647,0.1647
3,样本集幸存比例,1.0000,1.0000,0.0000,0.0000


## 8. 结论解读顺序

1. 先检查 OUT-OUT 与 IN-OUT 的阶段合计是否符合生产和本次置出规则预期。
2. 再把 IN-IN 当作“不置入”的基线，读取 `swap_result` 的变化前指标。
3. 最后加入 OUT-IN，读取 ALL-IN 的通过率、风险和金额变化。
4. OUT-IN 风险是评分预测值；生产落地前应对 `out_in_uplift` 做保守/中性/乐观情景测试。